# Main multi-run paired real-attack runner

This notebook uses a **shared prepared-staff cache**:

```text
dataset/cached_prepared_staffs/
```

Each run still gets its own outputs:

```text
results/runs/main_paired_<n_images>img_B<n_max_b>q/
```

For scientific comparability, smaller runs use nested subsets of the largest benchmark pool:

```text
25-image run ⊂ 50-image run ⊂ 100-image run
```

The notebook creates per-run lightweight cache views so Track B only attacks the selected subset, while the real prepared `.npy` files are reused from the shared cache.

## 1. Imports and repo setup

In [6]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Any


def now() -> str:
    return datetime.now().isoformat(timespec="seconds")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "attacks").exists() and (candidate / "dataset").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root. Launch the notebook from the repo or from attacks/.")


REPO_ROOT = find_repo_root()
SOURCE_IMAGES_DIR = REPO_ROOT / "dataset" / "images"
SHARED_CACHE_DIR = REPO_ROOT / "dataset" / "cached_prepared_staffs"

print("Repo root:", REPO_ROOT)
print("Source images:", SOURCE_IMAGES_DIR)
print("Shared prepared-staff cache:", SHARED_CACHE_DIR)


Repo root: D:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr
Source images: D:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr\dataset\images
Shared prepared-staff cache: D:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr\dataset\cached_prepared_staffs


## 2. Run configuration

In [7]:
# Add/remove configurations here.
# If run_tag is omitted, the folder name is auto-generated from n_images and n_max_b:
#   main_paired_<n_images>img_B<n_max_b>q

RUN_CONFIGS = [
    {
        "run_group": "main",
        "n_images": 100,
        "n_max_b": "20",
        "eps_b": ["0.0", "0.05", "0.10", "0.20", "0.30", "0.40"],
        "eps_a": ["0.0", "0.01", "0.02", "0.04", "0.08", "0.16", "0.32", "0.40"],
        "alpha_a": ["0", "1", "2"],
    },
    {
        "run_group": "main",
        "n_images": 25,
        "n_max_b": "100",
        "eps_b": ["0.0", "0.05", "0.10", "0.20", "0.30", "0.40"],
        "eps_a": ["0.0", "0.01", "0.02", "0.04", "0.08", "0.16", "0.32", "0.40"],
        "alpha_a": ["0", "1", "2"],
    },
    {
        "run_group": "main",
        "n_images": 50,
        "n_max_b": "50",
        "eps_b": ["0.0", "0.05", "0.10", "0.20", "0.30", "0.40"],
        "eps_a": ["0.0", "0.01", "0.02", "0.04", "0.08", "0.16", "0.32", "0.40"],
        "alpha_a": ["0", "1", "2"],
    },
]

P_INIT_B = "0.8"
P_FINAL_B = "0.05"

TARGET_SOURCE = "clean_onnx_prediction"
PLOT_SET = "core"
RESUME = True
CONTINUE_ON_ERROR = True

# Cache collection can try more source pages than max(n_images) if some pages fail.
CACHE_CANDIDATE_BATCH_SIZE = 25
MAX_CACHE_CANDIDATE_IMAGES = 500

# The shared benchmark pool is nested and reused by all configured runs.
MAX_N_IMAGES = max(int(cfg["n_images"]) for cfg in RUN_CONFIGS)

print("Configured runs:")
for cfg in RUN_CONFIGS:
    auto_tag = cfg.get("run_tag") or f"{cfg.get('run_group', 'main')}_paired_{int(cfg['n_images'])}img_B{cfg['n_max_b']}q"
    print(" -", auto_tag)
print("Largest nested benchmark pool:", MAX_N_IMAGES, "images")


Configured runs:
 - main_paired_100img_B20q
 - main_paired_25img_B100q
 - main_paired_50img_B50q
Largest nested benchmark pool: 100 images


## 3. Command runner and JSON helpers

In [8]:
def run_cmd(args: list[str], *, cwd: Path | None = None, log_path: Path | None = None) -> None:
    """Run a command with live output streamed into the notebook and optionally tee'd to a log file."""
    cwd = cwd or REPO_ROOT
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"

    cmd_display = " ".join(str(a) for a in args)
    print("\n$ " + cmd_display, flush=True)
    start = time.perf_counter()

    log_file = None
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_file = log_path.open("a", encoding="utf-8")
        log_file.write(f"\n[{now()}] $ {cmd_display}\n")
        log_file.flush()

    try:
        process = subprocess.Popen(
            [str(a) for a in args],
            cwd=str(cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None

        for line in process.stdout:
            print(line, end="", flush=True)
            if log_file is not None:
                log_file.write(line)

        return_code = process.wait()
        elapsed = time.perf_counter() - start
        print(f"[elapsed] {elapsed:.1f}s", flush=True)

        if log_file is not None:
            log_file.write(f"[elapsed] {elapsed:.1f}s\n")
            log_file.flush()

        if return_code != 0:
            raise RuntimeError(f"Command failed with exit code {return_code}: {cmd_display}")
    finally:
        if log_file is not None:
            log_file.close()


def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def safe_load_json(path: Path) -> dict[str, Any] | None:
    try:
        return load_json(path)
    except Exception:
        return None


def line_count(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


## 4. Artifact/progress helpers

In [9]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"}


def dedup_sorted_image_paths(images_dir: Path) -> list[Path]:
    """Return unique image paths in stable order. Needed on Windows because glob may be case-insensitive."""
    raw: list[Path] = []
    for ext in IMAGE_EXTENSIONS:
        raw.extend(images_dir.glob(f"*{ext}"))

    unique: dict[str, Path] = {}
    for path in raw:
        try:
            key = str(path.resolve()).casefold()
        except Exception:
            key = str(path.absolute()).casefold()
        unique[key] = path

    return sorted(unique.values(), key=lambda p: str(p).casefold())


def copy_or_hardlink(src: Path, dst: Path) -> None:
    """Prefer hardlink for speed/storage; fall back to copy."""
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return

    if dst.exists():
        dst.unlink()

    try:
        os.link(src, dst)
    except Exception:
        shutil.copy2(src, dst)


def mirror_tree_with_hardlinks(src_dir: Path, dst_dir: Path) -> None:
    """Create a lightweight copy of a cache subfolder using hardlinks when possible."""
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    for src in src_dir.rglob("*"):
        rel = src.relative_to(src_dir)
        dst = dst_dir / rel
        if src.is_dir():
            dst.mkdir(parents=True, exist_ok=True)
        else:
            copy_or_hardlink(src, dst)


def cache_dir_success(cache_image_dir: Path) -> bool:
    """A cached image directory is successful if it has metadata.json and at least one staff_*.npy."""
    if not cache_image_dir.exists():
        return False
    if not (cache_image_dir / "metadata.json").exists():
        return False
    return any(cache_image_dir.glob("staff_*.npy"))


def track_b_complete(track_b_json: Path) -> bool:
    data = safe_load_json(track_b_json)
    if not data:
        return False

    rows = data.get("aggregate_results") or data.get("square_attack_results") or []
    jobs = data.get("jobs", [])

    return len(rows) > 0 and len(jobs) > 0


def track_a_complete(track_a_json: Path) -> bool:
    data = safe_load_json(track_a_json)
    if not data:
        return False

    rows = data.get("aggregate_results") or data.get("spectral_noise_results") or []
    jobs = data.get("jobs", [])

    return len(rows) > 0 and len(jobs) > 0


def plotting_complete(plots_dir: Path) -> bool:
    required = [
        plots_dir / "spectral_delta_ser_vs_epsilon.html",
        plots_dir / "square_delta_ser_vs_epsilon.html",
        plots_dir / "combined_delta_ser_vs_normalized_severity.html",
        plots_dir / "combined_delta_cer_vs_normalized_severity.html",
        plots_dir / "combined_results_table.csv",
    ]
    return all(p.exists() and p.stat().st_size > 0 for p in required)


## 5. Ensure shared nested benchmark pool

In [10]:
def successful_images_in_source_order(source_images: list[Path]) -> list[Path]:
    """Return source images whose stem already has a successful shared-cache folder."""
    successful: list[Path] = []
    for image_path in source_images:
        if cache_dir_success(SHARED_CACHE_DIR / image_path.stem):
            successful.append(image_path)
    return successful


def ensure_shared_cache_pool(max_successful_images: int) -> list[Path]:
    """
    Ensure the shared cache contains enough successful pages for the largest configured run.

    The returned pool is in source order and is nested:
        first 25 ⊂ first 50 ⊂ first 100
    """
    SHARED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    source_images = dedup_sorted_image_paths(SOURCE_IMAGES_DIR)
    if len(source_images) < max_successful_images:
        raise RuntimeError(
            f"Requested {max_successful_images} successful pages, but only found {len(source_images)} unique source images."
        )

    candidate_count = min(max_successful_images, len(source_images), MAX_CACHE_CANDIDATE_IMAGES)

    while True:
        successful = successful_images_in_source_order(source_images[:candidate_count])
        print(f"[shared-cache] successful pages in first {candidate_count} candidates: {len(successful)}/{max_successful_images}")

        if len(successful) >= max_successful_images:
            pool = successful[:max_successful_images]
            print("[shared-cache] enough cached pages available")
            return pool

        max_candidates = min(len(source_images), MAX_CACHE_CANDIDATE_IMAGES)
        if candidate_count >= max_candidates:
            raise RuntimeError(
                f"Could only obtain {len(successful)} successful cached pages after trying {candidate_count} candidates. "
                f"Check dataset/cached_prepared_staffs and cache logs."
            )

        # Expand candidate window, then ask the cache script to process that window.
        candidate_count = min(candidate_count + CACHE_CANDIDATE_BATCH_SIZE, max_candidates)

        run_cmd(
            [
                sys.executable,
                "dataset/cache_prepared_staffs.py",
                "--images-dir",
                str(SOURCE_IMAGES_DIR),
                "--output-dir",
                str(SHARED_CACHE_DIR),
                "--model-path",
                "models/onnx/segnet.onnx",
                "--limit",
                str(candidate_count),
                "--continue-on-error",
            ],
            log_path=REPO_ROOT / "results" / "runs" / "_shared_cache_logs" / "cache_prepared_staffs_live.log",
        )


BENCHMARK_IMAGE_POOL = ensure_shared_cache_pool(MAX_N_IMAGES)

POOL_PATH = REPO_ROOT / "results" / "runs" / "main_benchmark_pool.txt"
POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
POOL_PATH.write_text("\n".join(str(p.relative_to(REPO_ROOT)) for p in BENCHMARK_IMAGE_POOL) + "\n", encoding="utf-8")

print("Benchmark pool:", POOL_PATH)
for p in BENCHMARK_IMAGE_POOL[:5]:
    print(" -", p.relative_to(REPO_ROOT))
if len(BENCHMARK_IMAGE_POOL) > 5:
    print(" ...", len(BENCHMARK_IMAGE_POOL) - 5, "more")


[shared-cache] successful pages in first 100 candidates: 14/100

$ c:\Users\theda\miniconda3\envs\adv-homr\python.exe dataset/cache_prepared_staffs.py --images-dir D:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr\dataset\images --output-dir D:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr\dataset\cached_prepared_staffs --model-path models/onnx/segnet.onnx --limit 125 --continue-on-error
SegNet providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
SegNet input: input ['batch_size', 3, 320, 320]
SegNet output: output ['batch_size', 6, 320, 320]

[1/125] Qmb615Rgf2hhA2mdiyx4q3RUCCyUpwZfgBoNVJ7bAskCPx-1.png
  skipped_existing: 6 prepared staff image(s)

[2/125] Qmb616rpqZKrYzjs5FwqQQ8FxeHg8eSbjzoQugG5RPnQJx-1.png
  skipped_existing: 12 prepared staff image(s)

[3/125] Qmb616rpqZKrYzjs5FwqQQ8FxeHg8eSbjzoQugG5RPnQJx-2.png
  skipped_existing: 4 prepared staff image(s)

[4/125] Qmb616rpqZKrYzjs5Fwq

## 6. Build per-run context

In [11]:
def build_context(cfg: dict[str, Any]) -> dict[str, Any]:
    n_images = int(cfg["n_images"])
    n_max_b = str(cfg["n_max_b"])

    run_group = str(cfg.get("run_group", "main"))
    base_run_id = f"{run_group}_paired_{n_images}img_B{n_max_b}q"

    # Optional manual override. By default the run ID is generated from the numbers.
    run_tag = cfg.get("run_tag")
    run_id = base_run_id if run_tag in (None, "") else str(run_tag)

    run_dir = REPO_ROOT / "results" / "runs" / run_id
    logs_dir = run_dir / "logs"
    plots_dir = run_dir / "plots"
    status_dir = run_dir / "status"

    # Per-run outputs/views. These are lightweight and only include the selected subset.
    final_images_dir = run_dir / "paired_images"
    cache_view_dir = run_dir / "cached_prepared_staffs_view"

    for path in [run_dir, logs_dir, plots_dir, status_dir, final_images_dir, cache_view_dir]:
        path.mkdir(parents=True, exist_ok=True)

    return {
        "cfg": cfg,
        "run_id": run_id,
        "n_images": n_images,
        "n_max_b": n_max_b,
        "run_dir": run_dir,
        "logs_dir": logs_dir,
        "plots_dir": plots_dir,
        "status_dir": status_dir,
        "final_images_dir": final_images_dir,
        "cache_view_dir": cache_view_dir,
        "manifest_path": run_dir / "run_manifest.json",
        "image_subset_path": run_dir / "image_subset.txt",
        "staff_list_path": run_dir / "cached_staffs.txt",
        "subset_complete_path": status_dir / "subset_view_complete.json",
        "track_b_json": logs_dir / f"TRACK_B_{run_id}_metrics.json",
        "track_b_checkpoint": logs_dir / f"TRACK_B_{run_id}_checkpoint.jsonl",
        "track_b_timing": logs_dir / f"TRACK_B_{run_id}_timing.jsonl",
        "track_a_json": logs_dir / f"TRACK_A_{run_id}_metrics.json",
        "track_a_checkpoint": logs_dir / f"TRACK_A_{run_id}_checkpoint.jsonl",
        "track_a_timing": logs_dir / f"TRACK_A_{run_id}_timing.jsonl",
    }


## 7. Create run-specific image subset and cache view

In [12]:
def subset_view_complete(ctx: dict[str, Any], selected_images: list[Path]) -> bool:
    marker = safe_load_json(ctx["subset_complete_path"])
    if not marker:
        return False

    if marker.get("n_images") != ctx["n_images"]:
        return False

    if not ctx["image_subset_path"].exists() or not ctx["staff_list_path"].exists():
        return False

    if len(list(ctx["final_images_dir"].glob("*"))) < ctx["n_images"]:
        return False

    if len(list(ctx["cache_view_dir"].rglob("staff_*.npy"))) <= 0:
        return False

    expected_stems = [p.stem for p in selected_images]
    actual_stems = sorted([p.name for p in ctx["cache_view_dir"].iterdir() if p.is_dir()])
    return sorted(expected_stems) == actual_stems


def prepare_run_subset_and_cache_view(ctx: dict[str, Any], benchmark_pool: list[Path]) -> None:
    cfg = ctx["cfg"]
    selected_images = benchmark_pool[: ctx["n_images"]]

    if subset_view_complete(ctx, selected_images):
        print("[skip] run subset/cache view already complete")
        return

    # Rebuild view folders. These are only views/copies of already cached shared artifacts.
    if ctx["final_images_dir"].exists():
        shutil.rmtree(ctx["final_images_dir"])
    if ctx["cache_view_dir"].exists():
        shutil.rmtree(ctx["cache_view_dir"])
    ctx["final_images_dir"].mkdir(parents=True, exist_ok=True)
    ctx["cache_view_dir"].mkdir(parents=True, exist_ok=True)

    image_lines: list[str] = []
    for image_path in selected_images:
        shared_cache_image_dir = SHARED_CACHE_DIR / image_path.stem
        if not cache_dir_success(shared_cache_image_dir):
            raise RuntimeError(f"Selected image is not available in shared cache: {image_path}")

        dst_img = ctx["final_images_dir"] / image_path.name
        copy_or_hardlink(image_path, dst_img)
        image_lines.append(str(dst_img.relative_to(REPO_ROOT)))

        mirror_tree_with_hardlinks(shared_cache_image_dir, ctx["cache_view_dir"] / image_path.stem)

    ctx["image_subset_path"].write_text("\n".join(image_lines) + "\n", encoding="utf-8")

    staff_files = sorted(ctx["cache_view_dir"].rglob("staff_*.npy"))
    ctx["staff_list_path"].write_text(
        "\n".join(str(p.relative_to(REPO_ROOT)) for p in staff_files) + "\n",
        encoding="utf-8",
    )

    manifest = {
        "run_id": ctx["run_id"],
        "updated": now(),
        "repo_root": str(REPO_ROOT),
        "shared_cache_dir": str(SHARED_CACHE_DIR),
        "n_images": ctx["n_images"],
        "n_max_b": ctx["n_max_b"],
        "eps_a": cfg["eps_a"],
        "alpha_a": cfg["alpha_a"],
        "eps_b": cfg["eps_b"],
        "p_init_b": P_INIT_B,
        "p_final_b": P_FINAL_B,
        "target_source": TARGET_SOURCE,
        "plot_set": PLOT_SET,
        "run_dir": str(ctx["run_dir"]),
        "final_images_dir": str(ctx["final_images_dir"]),
        "cache_view_dir": str(ctx["cache_view_dir"]),
        "image_subset_path": str(ctx["image_subset_path"]),
        "staff_list_path": str(ctx["staff_list_path"]),
        "staff_count": len(staff_files),
        "nested_subset_policy": "first_n_images_from_shared_benchmark_pool",
    }
    save_json(ctx["manifest_path"], manifest)

    save_json(
        ctx["subset_complete_path"],
        {
            "completed_at": now(),
            "n_images": ctx["n_images"],
            "staff_count": len(staff_files),
            "image_subset_path": str(ctx["image_subset_path"]),
            "staff_list_path": str(ctx["staff_list_path"]),
        },
    )

    print("[done] run subset/cache view complete")
    print("Images:", ctx["n_images"])
    print("Staffs:", len(staff_files))


## 8. Track B

In [13]:
def run_track_b(ctx: dict[str, Any]) -> None:
    if track_b_complete(ctx["track_b_json"]):
        print("[skip] Track B already complete:", ctx["track_b_json"])
        return

    cfg = ctx["cfg"]

    cmd = [
        sys.executable,
        "attacks/run_square_sweep.py",
        "--cache-dir",
        str(ctx["cache_view_dir"]),
        "--epsilon",
        *cfg["eps_b"],
        "--n-max",
        ctx["n_max_b"],
        "--p-init",
        P_INIT_B,
        "--p-final",
        P_FINAL_B,
        "--target-source",
        TARGET_SOURCE,
        "--progress-every",
        "1",
        "--checkpoint-jsonl",
        str(ctx["track_b_checkpoint"]),
        "--timing-jsonl",
        str(ctx["track_b_timing"]),
        "--output-json",
        str(ctx["track_b_json"]),
    ]

    if RESUME:
        cmd.append("--resume")

    if CONTINUE_ON_ERROR:
        cmd.append("--continue-on-error")

    run_cmd(cmd, log_path=ctx["logs_dir"] / "03_track_b_live.log")


## 9. Track A

In [14]:
def run_track_a(ctx: dict[str, Any]) -> None:
    if track_a_complete(ctx["track_a_json"]):
        print("[skip] Track A already complete:", ctx["track_a_json"])
        return

    cfg = ctx["cfg"]

    cmd = [
        sys.executable,
        "attacks/run_spectral_sweep.py",
        "--images-dir",
        str(ctx["final_images_dir"]),
        "--max-images",
        str(ctx["n_images"]),
        "--epsilon",
        *cfg["eps_a"],
        "--alpha",
        *cfg["alpha_a"],
        "--target-source",
        TARGET_SOURCE,
        "--output-json",
        str(ctx["track_a_json"]),
        "--timing-jsonl",
        str(ctx["track_a_timing"]),
        "--checkpoint-jsonl",
        str(ctx["track_a_checkpoint"]),
    ]

    if RESUME:
        cmd.append("--resume")

    if CONTINUE_ON_ERROR:
        cmd.append("--continue-on-error")

    run_cmd(cmd, log_path=ctx["logs_dir"] / "04_track_a_live.log")


## 10. Plotting

In [15]:
def run_plots(ctx: dict[str, Any]) -> None:
    if plotting_complete(ctx["plots_dir"]):
        print("[skip] plotting already complete")
        return

    run_cmd(
        [
            sys.executable,
            str(REPO_ROOT / "attacks" / "python scripts" / "plot_results.py"),
            "--spectral-json",
            str(ctx["track_a_json"]),
            "--square-json",
            str(ctx["track_b_json"]),
            "--combined",
            "--plot-set",
            PLOT_SET,
            "--output-dir",
            str(ctx["plots_dir"]),
        ],
        log_path=ctx["logs_dir"] / "05_plotting_live.log",
    )


## 11. Summary

In [16]:
def summarize_run(ctx: dict[str, Any]) -> None:
    summary_paths = [
        ctx["manifest_path"],
        ctx["image_subset_path"],
        ctx["staff_list_path"],
        ctx["subset_complete_path"],
        ctx["track_b_json"],
        ctx["track_b_checkpoint"],
        ctx["track_b_timing"],
        ctx["track_a_json"],
        ctx["track_a_checkpoint"],
        ctx["track_a_timing"],
        ctx["plots_dir"] / "combined_results_table.csv",
        ctx["plots_dir"] / "combined_delta_ser_vs_normalized_severity.html",
        ctx["plots_dir"] / "combined_delta_cer_vs_normalized_severity.html",
    ]

    for path in summary_paths:
        print(path)
        print("  exists:", path.exists())
        if path.exists():
            print("  size:", path.stat().st_size, "bytes")


## 12. Run all configured combinations

In [ ]:
all_contexts = []

for index, cfg in enumerate(RUN_CONFIGS, start=1):
    ctx = build_context(cfg)
    all_contexts.append(ctx)

    print("\n" + "=" * 88)
    print(f"RUN {index}/{len(RUN_CONFIGS)}: {ctx['run_id']}")
    print("=" * 88)

    prepare_run_subset_and_cache_view(ctx, BENCHMARK_IMAGE_POOL)
    run_track_b(ctx)
    run_track_a(ctx)
    run_plots(ctx)
    summarize_run(ctx)

print("\nAll configured runs finished or skipped as complete.")



RUN 1/3: main_paired_100img_B20q
[done] run subset/cache view complete
Images: 100
Staffs: 670

$ c:\Users\theda\miniconda3\envs\adv-homr\python.exe attacks/run_square_sweep.py --cache-dir D:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr\results\runs\main_paired_100img_B20q\cached_prepared_staffs_view --epsilon 0.0 0.05 0.10 0.20 0.30 0.40 --n-max 20 --p-init 0.8 --p-final 0.05 --target-source clean_onnx_prediction --progress-every 1 --checkpoint-jsonl D:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr\results\runs\main_paired_100img_B20q\logs\TRACK_B_main_paired_100img_B20q_checkpoint.jsonl --timing-jsonl D:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr\results\runs\main_paired_100img_B20q\logs\TRACK_B_main_paired_100img_B20q_timing.jsonl --output-json D:\Users\theda\Documents\dev\Projects\Uni\Semester 4\Artificial Intelligence\adversarial-homr\results\